# Day 14 — Same agent, five ways: your loop, automatic function calling, LangGraph, OpenAI Agents SDK, Google ADK
**Theory:** [day-14-frameworks.md](../theory/day-14-frameworks.md) · ⏱ ~105 min · Level: medium

Every section uses Gemini and is wrapped in `try/except` (and a timeout for async ones), so one framework's
install problem doesn't stop the others. Frameworks move fast. If an import fails, check that framework's
quickstart and pin a version.

In [ ]:
# ⚙️ Setup: works in GitHub Codespaces, locally, in Colab and on Kaggle. Run this cell first.
import os, sys, pathlib, subprocess
IN_COLAB, IN_KAGGLE = "google.colab" in sys.modules, os.path.exists("/kaggle/working")
if IN_COLAB or IN_KAGGLE:  # clone + install the course repo, read the key from notebook secrets
    if not pathlib.Path("agentic-ai-course").exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/Rahulverma5/agentic-ai-course"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "agentic-ai-course/requirements.txt"], check=True)
    try:
        if IN_COLAB:
            from google.colab import userdata
            os.environ.setdefault("GEMINI_API_KEY", userdata.get("GEMINI_API_KEY"))
        else:
            from kaggle_secrets import UserSecretsClient
            os.environ.setdefault("GEMINI_API_KEY", UserSecretsClient().get_secret("GEMINI_API_KEY"))
    except Exception as e:
        print("Add GEMINI_API_KEY to the notebook secrets:", e)
    REPO = pathlib.Path("agentic-ai-course").resolve()
else:  # Codespaces / local: find the repo root
    REPO = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path[:0] = [str(REPO), str(REPO / "projects" / "shopease-support-agent")]
os.chdir(REPO)  # every path in this notebook is relative to the repo root
from google.genai import types  # Gemini SDK types: Content, Part, GenerateContentConfig, ...
from course_utils import check_setup, client, MODEL, CHEAP_MODEL, ask, tracker, text_of, thinking, user
from shopease.data import setup_data
setup_data(quiet=True)  # creates the ShopEase sample data if it's missing
check_setup();

In [ ]:
# Install framework packages (only needed today; ~1–3 min in Codespaces)
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-frameworks.txt"], check=False)

In [ ]:
import json, time, asyncio, os, traceback
import pandas as pd
from shopease.tools import Session, get_order as _get_order, check_refund_eligibility as _check
os.environ.setdefault("GOOGLE_API_KEY", os.environ.get("GEMINI_API_KEY", ""))  # some frameworks read GOOGLE_API_KEY
session = Session("C001")
QUESTION = "Can I get a refund for my coffee grinder O2002?"
results = []
def record(name, fn):
    t0 = time.time()
    try:
        answer = fn()
    except Exception as e:
        answer = f"FAILED: {type(e).__name__}: {e}"
        traceback.print_exc(limit=1)
    results.append({"framework": name, "seconds": round(time.time() - t0, 1), "answer": str(answer)[:200]})
    print(f"[{name}] {answer}\n")

# Plain Python tools: docstring + type hints become the schema in every framework below.
def get_order(order_id: str) -> str:
    """Look up one of the customer's ShopEase orders by id, e.g. 'O2001'."""
    return _get_order(session, order_id)

def check_refund_eligibility(order_id: str) -> str:
    """Check if an order can be refunded under ShopEase policy."""
    return _check(session, order_id)

## 0. Baseline: your own loop (course_utils.run_agent)

In [ ]:
from course_utils import run_agent
from shopease.tools import TOOLS
two_tools = [t for t in TOOLS if t["name"] in ("get_order", "check_refund_eligibility")]
fns = {"get_order": get_order, "check_refund_eligibility": check_refund_eligibility}
record("raw SDK loop", lambda: run_agent(QUESTION, two_tools, fns, thinking_level="low", verbose=False).text)

## 1. google-genai automatic function calling: the SDK runs the loop
Pass Python functions as tools and the SDK declares them, calls them and loops until the model answers.

In [ ]:
def automatic_function_calling():
    r = client.models.generate_content(model=MODEL, contents=QUESTION, config=types.GenerateContentConfig(
        tools=[get_order, check_refund_eligibility],
        automatic_function_calling=types.AutomaticFunctionCallingConfig(maximum_remote_calls=6)))
    tracker.add(MODEL, r.usage_metadata, "AFC")
    steps = len(r.automatic_function_calling_history or [])
    return f"({steps} history entries) {text_of(r)}"
record("google-genai automatic function calling", automatic_function_calling)

## 2. LangGraph: the loop as an explicit graph

In [ ]:
def langgraph_agent():
    from langchain_google_genai import ChatGoogleGenerativeAI
    from langchain_core.tools import tool
    from langgraph.graph import StateGraph, MessagesState, START
    from langgraph.prebuilt import ToolNode, tools_condition

    tools = [tool(get_order), tool(check_refund_eligibility)]
    llm = ChatGoogleGenerativeAI(model=MODEL, google_api_key=os.environ.get("GEMINI_API_KEY")).bind_tools(tools)
    g = StateGraph(MessagesState)
    g.add_node("agent", lambda state: {"messages": [llm.invoke(state["messages"])]})
    g.add_node("tools", ToolNode(tools))
    g.add_edge(START, "agent")
    g.add_conditional_edges("agent", tools_condition)   # → "tools" or END
    g.add_edge("tools", "agent")
    app = g.compile()
    out = app.invoke({"messages": [("user", QUESTION)]})
    return out["messages"][-1].content
record("LangGraph", langgraph_agent)

## 3. OpenAI Agents SDK (with Gemini via its LiteLLM extension)

In [ ]:
async def openai_agents_sdk():
    from agents import Agent, Runner, function_tool, set_tracing_disabled
    from agents.extensions.models.litellm_model import LitellmModel
    set_tracing_disabled(True)  # its default tracing exports to OpenAI; we have no OpenAI key here

    agent = Agent(name="ShopEase support", instructions="Help the customer with their orders. Check eligibility before refunds.",
                  tools=[function_tool(get_order), function_tool(check_refund_eligibility)],
                  model=LitellmModel(model=f"gemini/{CHEAP_MODEL}", api_key=os.environ.get("GEMINI_API_KEY")))
    result = await Runner.run(agent, QUESTION)
    return result.final_output

t0 = time.time()
try:
    ans = await asyncio.wait_for(openai_agents_sdk(), timeout=180)
except Exception as e:
    ans = f"FAILED: {type(e).__name__}: {e}"
results.append({"framework": "OpenAI Agents SDK (Gemini via LiteLLM)", "seconds": round(time.time() - t0, 1), "answer": str(ans)[:200]})
print(ans)

## 4. Google Agent Development Kit (ADK): Google's agent framework
Agents, tools, sessions, multi-agent and deployment, built around Gemini. Tools are plain Python functions.

In [ ]:
async def google_adk():
    from google.adk.agents import Agent
    from google.adk.runners import InMemoryRunner

    agent = Agent(name="shopease_support", model=MODEL,
                  instruction="You are ShopEase support. Check refund eligibility before answering refund questions.",
                  tools=[get_order, check_refund_eligibility])
    runner = InMemoryRunner(agent=agent, app_name="shopease")
    s = await runner.session_service.create_session(app_name="shopease", user_id="u1")
    final = ""
    async for event in runner.run_async(user_id="u1", session_id=s.id, new_message=user(QUESTION)):
        if event.is_final_response() and event.content and event.content.parts:
            final = "".join(p.text or "" for p in event.content.parts)
    return final

t0 = time.time()
try:
    ans = await asyncio.wait_for(google_adk(), timeout=180)
except Exception as e:
    ans = f"FAILED: {type(e).__name__}: {e}"
results.append({"framework": "Google ADK", "seconds": round(time.time() - t0, 1), "answer": str(ans)[:200]})
print(ans)

## 5. Compare

In [ ]:
pd.DataFrame(results)

**Write a 1-page framework decision record (ADR):** context, options, decision, consequences. Use the table
above plus: lines of code, how easy it was to see what was sent to the model, persistence, human-in-the-loop, tracing.

## ✏️ Exercises
1. Add `interrupt_before=["tools"]` to the LangGraph compile step to get human approval before tools run.
2. In ADK, add a second agent (refunds specialist) as a sub-agent of a triage agent.
3. Run the same 5 eval cases through two frameworks and compare pass rates.

## 🧾 What did this notebook cost?
Every call above was recorded by `tracker` (estimated at paid prices; the free tier costs $0). Make a habit of checking tokens and cost after every experiment.

In [ ]:
print(tracker.summary())
tracker.to_dataframe().tail(10) if tracker.calls else None